In [29]:
#Importing necessary libraries
from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface  import HuggingFaceEmbeddings 
from langchain_core.prompts import ChatPromptTemplate
from typing import List
from langchain_classic.schema import Document
from pinecone import Pinecone
import os
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
### Data Loading 
def load_pdf_files(pdf_path):
    print(f"Loading pdf files from {pdf_path}...")
    loader=DirectoryLoader(pdf_path,glob="**/*.pdf",
                           show_progress=True,
                           loader_cls=PyMuPDFLoader)
    documents=loader.load()
    print(f"Finished loading pdf files from {pdf_path}.")
    return documents

In [12]:
# Removing unnecessary metadata from the documents to save memory and focus on content and source.

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [13]:
# Text Splitting 
def split_documents(minimal_documents, chunk_size: int = 500, chunk_overlap: int = 100) :
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    text_chunks = text_splitter.split_documents(minimal_documents)
    return text_chunks

In [3]:
pc=Pinecone()
pc

In [ ]:
from pinecone import ServerlessSpec
index_name="multimodal-agentic-app"
if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [14]:
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5629.50it/s]


In [21]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings_model,
    index_name=index_name
)

In [22]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings_model
)

In [23]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [25]:
retrieved_docs = retriever.invoke("What is protein requirementfor adults?")
retrieved_docs

[Document(id='0b2769ca-cc9d-4efe-9727-142ca9d4dcfe', metadata={'source': '..\\pdf_data\\food_nutrtion.pdf'}, page_content='Food  a n d  Nu t r i t i on  Ha n d b ook  fo r  Exte nsi o n Wo r ke rs\n6\nKEY MESSAGE \nExtra protein is required during illness, convalescence and after \nsurgery because the body has extra demands for protein to replace \nand repair worn out tissues.\nProtein requirements\nThe recommended intake of protein each day is about 1 gram per kilo\xad\ngram of body weight. Example: if a person is 60 kilograms, he will require \n60 grams of protein each day. This is equivalent to one egg or a piece of'),
 Document(id='48ba04f2-6716-4cfb-bd86-06b93c7eca7a', metadata={'source': '..\\pdf_data\\food_nutrtion.pdf'}, page_content='60 grams of protein each day. This is equivalent to one egg or a piece of \nmeat about the size of an egg. However, children, teenagers, and preg\xad\nnant and lactating mothers require more protein as indicated below:\n•\t Children: 30–50 g (half

In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [30]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [55]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain with proper document formatting
rag_chain = (
    {
        "context": itemgetter("input") | retriever | format_docs,
        "input": itemgetter("input")
    }
    | prompt
    | chatModel
    | StrOutputParser()
)

In [56]:
rag_chain.invoke({"input": "what is protein requirement for adults?"})

'Adults require 60–70 grams of protein daily, which is roughly equivalent to a palm of meat. Generally, the recommended intake is about 1 gram per kilogram of body weight. For instance, a 60-kilogram person would need 60 grams of protein each day.'